# 董事會財務分析：各季各家第一場法說會

> ⚠️ **此 notebook 已併入 `產生法說會查詢網站.py`，僅供分析參考、不再維護。**
> 正式流程請執行 `產生法說會查詢網站.py`（讀 CSV → 即時爬蟲 → 分析 → 產生網站），
> 它已包含：合併 `公告財報.csv`、即將法說(upcoming)進分析、每季早上/下午 + 外資/內資主辦分類。

合併董事會通過/決議 → 解析財報期別 → 逐年截止日 → 以「公司分組、依法說當下已通過的最近一季」判定每季第一場法說（互斥；排除展望季）→ 存 `財報_df.pkl` → 涵蓋率檢查。

In [1]:
import re, glob
from datetime import date
import numpy as np
import pandas as pd
import exchange_calendars as xcals

BASE = r'E:\法說會+主動型'
xtai = xcals.get_calendar('XTAI')

## 1. 合併董事會通過 + 決議

In [2]:
通過_df = pd.read_csv(rf'{BASE}\董事會通過.csv', encoding='utf-8-sig')
決議_df = pd.read_csv(rf'{BASE}\董事會決議.csv', encoding='utf-8-sig')
通過_df['來源'] = '通過'
決議_df['來源'] = '決議'
董事會_df = pd.concat([通過_df, 決議_df], ignore_index=True)
print(f'董事會通過 {len(通過_df):,} + 決議 {len(決議_df):,} = 合併 {len(董事會_df):,} 筆')

董事會通過 28,628 + 決議 7,011 = 合併 35,639 筆


## 2. 解析財報期別（財報民國年 + 期別）

In [3]:
_CN = {'一':'1','ㄧ':'1','二':'2','三':'3','四':'4','五':'5',
       '六':'6','七':'7','八':'8','九':'9','〇':'0','○':'0','零':'0','０':'0'}
def cn_year(s): return int(''.join(_CN[c] for c in s if c in _CN))
_Q_CN = {'一':'Q1','二':'Q2','三':'Q3','四':'Q4','1':'Q1','2':'Q2','3':'Q3','4':'Q4'}

def extract_roc_year(title, fallback):
    fallback = int(fallback)
    m = re.search(r'\((\d{3})年\)', title)
    if m: return int(m.group(1))
    m = re.search(r'(?<!\d)(\d{3})(?!\d)年', title)
    if m: return int(m.group(1))
    m = re.search(r'(20[012]\d)年', title)
    if m: return int(m.group(1)) - 1911
    _cn3 = r'[一ㄧ二三四五六七八九〇○零０]{3}'
    m = re.search(rf'民國({_cn3})年', title)
    if m: return cn_year(m.group(1))
    _any = r'[一二三四五六七八九〇○零０ㄧ]'
    m = re.search(rf'([一ㄧ]{_any}{_any})年', title)
    if m:
        y = cn_year(m.group(1))
        if 100 <= y <= 130: return y
    return fallback

def extract_period(title):
    m = re.search(r'第([一二三四1234])季', title)
    if m: return _Q_CN.get(m.group(1))
    m = re.search(r'Q([1234])', title, re.IGNORECASE)
    if m: return f'Q{m.group(1)}'
    if re.search(r'年度|全年|(\d{3})度', title): return '年報'
    if re.search(r'(?<!\d)\d{3}(?!\d)年|民國|[一ㄧ][〇○一]', title): return '年報'
    return '其他'

_EXCLUDE = re.compile(r'主管異動|背書保證|財務預測|財務危機|財測|財務長|財務副|帳務|財物|重編|資金貸與|現金貸')
_INCLUDE = re.compile(r'財務報告|財務報表|財報')

def parse_report_period(title, announce_roc_year):
    t = str(title).replace('\r\n', ' ').replace('\n', ' ')
    if _EXCLUDE.search(t): return None, None
    if not _INCLUDE.search(t): return None, None
    announce_roc_year = int(announce_roc_year)
    fy = extract_roc_year(t, announce_roc_year)
    period = extract_period(t)
    if abs(fy - announce_roc_year) > 2: fy = announce_roc_year
    if period in ('Q4', '年報') and fy >= announce_roc_year: fy = announce_roc_year - 1
    return fy, period

董事會_df[['財報民國年', '期別']] = 董事會_df.apply(
    lambda r: pd.Series(parse_report_period(r['主旨'], r['民國年'])), axis=1)
財報_raw = 董事會_df.dropna(subset=['期別']).copy()
財報_raw = 財報_raw[財報_raw['期別'] != '其他'].copy()
財報_raw['財報民國年'] = 財報_raw['財報民國年'].astype(int)
print(f'解析出財報公告 {len(財報_raw):,} 筆（期別分布）')
print(財報_raw['期別'].value_counts().to_dict())

解析出財報公告 34,700 筆（期別分布）
{'年報': 9315, 'Q1': 9051, 'Q2': 7628, 'Q3': 7433, 'Q4': 1273}


## 3. 大型股 + 逐年截止日

In [4]:
cap_df = pd.read_csv(rf'{BASE}\實收資本額.csv', encoding='utf-16', sep='\t')
cap_df['代號'] = cap_df['證券代碼'].str.extract(r'^(\d+)').astype(int)
cap_df['大型股'] = cap_df['實收資本額(元)'] >= 10_000_000_000
large_cap_set = set(cap_df.loc[cap_df['大型股'], '代號'])
財報_raw['大型股'] = 財報_raw['代號'].isin(large_cap_set)

# 大型股年報實際截止日：key = 財報民國年 → (月,日) 於隔年
_LC_ANNUAL = {111: (3, 16), 112: (3, 15), 113: (3, 17), 114: (3, 16)}
def lc_annual_deadline(fy_roc):
    m, d = _LC_ANNUAL.get(fy_roc, (3, 16))
    return date(fy_roc + 1912, m, d)

def get_deadline(period, fy_roc, large):
    fy_ce = fy_roc + 1911
    if period == 'Q1': return date(fy_ce, 5, 15)
    if period == 'Q2': return date(fy_ce, 8, 14)
    if period == 'Q3': return date(fy_ce, 11, 14)
    if period == 'Q4': return date(fy_ce + 1, 3, 31)
    if period == '年報': return lc_annual_deadline(fy_roc) if large else date(fy_ce + 1, 4, 1)
    return None

def roc_to_date(s):
    try:
        y, m, d = map(int, str(s).split('/')); return date(y + 1911, m, d)
    except: return None

def parse_board_date(row):
    t = str(row['主旨']).replace('\r\n', ' ').replace('\n', ' ')
    m = re.search(r'於(\d{3})年(\d{1,2})月(\d{1,2})日', t)
    if m:
        try:
            dt = date(int(m.group(1)) + 1911, int(m.group(2)), int(m.group(3)))
            filing = roc_to_date(row['日期'])
            if filing and abs((dt - filing).days) <= 60: return dt
        except: pass
    return roc_to_date(row['日期'])

財報_raw['截止日'] = 財報_raw.apply(lambda r: get_deadline(r['期別'], r['財報民國年'], r['大型股']), axis=1)
財報_raw['通過日'] = 財報_raw.apply(parse_board_date, axis=1)
財報_raw = 財報_raw.dropna(subset=['通過日', '截止日']).copy()

## 4. 正規化：每 (代號, 財報年, 期別) 取最早通過日為錨點（合併通過/決議重複公告）

In [5]:
財報_raw = 財報_raw.sort_values('通過日')
季報_df = (財報_raw
    .groupby(['代號', '財報民國年', '期別'], as_index=False)
    .agg(簡稱=('簡稱', 'first'), 市場別=('市場別', 'first'),
         大型股=('大型股', 'first'), 截止日=('截止日', 'first'),
         通過日=('通過日', 'first'), 主旨=('主旨', 'first')))
print(f'正規化後各季各家財報列：{len(季報_df):,} 筆')

正規化後各季各家財報列：33,579 筆


## 5. 匯入法說會（上市 + 上櫃），解析法說日 + 擇要「報告」財報期別
（排除「業績展望/預期」的未來季別，避免把展望季當成本場財報季）

In [6]:
def load_inv(folder):
    parts = []
    for f in sorted(glob.glob(rf'{BASE}\{folder}\*.csv')):
        tmp = pd.read_csv(f, encoding='cp950', encoding_errors='ignore', on_bad_lines='skip')
        parts.append(tmp)
    return pd.concat(parts, ignore_index=True)

inv = pd.concat([load_inv('上市法說會'), load_inv('上櫃法說會')], ignore_index=True)
def parse_inv_date(s):
    try:
        y, m, d = map(int, str(s).split('/')); return date(y + 1911, m, d)
    except: return None
inv['法說日'] = inv['召開法人說明會日期'].apply(parse_inv_date)
inv = inv.dropna(subset=['法說日']).copy()
inv['代號'] = inv['公司代號'].astype(str).str.strip()
print(f'法說會 {len(inv):,} 筆')

_Q_T = {'一':'Q1','二':'Q2','三':'Q3','四':'Q4','1':'Q1','2':'Q2','3':'Q3','4':'Q4'}
# 展望/預期句型：把「(年)第X季(業績)展望/預期/預估」的季別視為展望，不算本場財報期
_OUTLOOK_RE = re.compile(r'(20\d{2}|1[01]\d)?年?第([一二三四1-4])季(?:業績|營運|獲利)?(?:展望|預期|預估|預測)')

def extract_reported_periods(text, conf_date):
    """回傳本場法說『報告』的財報期別 set[(財報民國年, 期別)]，排除展望季。"""
    if pd.isna(text) or not str(text).strip(): return set()
    t = str(text)
    allp = set()
    for y, q in re.findall(r'(20\d{2})年度?第([一二三四1-4])季', t): allp.add((int(y) - 1911, _Q_T[q]))
    for y, q in re.findall(r'(1[01]\d)年度?第([一二三四1-4])季', t): allp.add((int(y), _Q_T[q]))
    for y, q in re.findall(r'(20\d{2})[Qq]([1-4])', t): allp.add((int(y) - 1911, f'Q{q}'))
    for y, q in re.findall(r'(1[01]\d)[Qq]([1-4])', t): allp.add((int(y), f'Q{q}'))
    for y in re.findall(r'(20\d{2})年?(?:度|全年|年報)', t): allp.add((int(y) - 1911, '年報'))
    for y in re.findall(r'(1[01]\d)年度', t): allp.add((int(y), '年報'))
    if conf_date is not None:
        yr = conf_date.year - 1911
        for q in re.findall(r'第([一二三四1-4])季', t):
            qs = _Q_T[q]; allp.add((yr - 1 if qs == 'Q4' else yr, qs))
    # 移除展望季
    outlook = set()
    for m in _OUTLOOK_RE.finditer(t):
        y = m.group(1); qs = _Q_T[m.group(2)]
        if y: fy = int(y) - 1911 if int(y) >= 1911 else int(y)
        elif conf_date is not None: fy = conf_date.year - 1911
        else: fy = None
        if fy is not None: outlook.add((fy, qs))
    rep = allp - outlook
    return rep if rep else allp        # 若整段只有展望季，仍保留以免全空

inv['擇要報告期'] = inv.apply(
    lambda r: extract_reported_periods(r.get('法人說明會擇要訊息'), r['法說日']), axis=1)

法說會 21,989 筆


## 6. 各季第一場法說：以「法說當下已通過的最近一季」歸屬（互斥）
規則（依使用者定義）：法說歸屬於『法說日當下董事會已通過的最近一季』。
· 擇要明確『報告』到尚未通過的下一季(且非展望) → 前置法說(台積電模式)。
· 同一場法說只會屬於一季；每季取最早一場。

In [7]:
def assign_quarter(comp_q, comp_inv):
    q = comp_q.sort_values('通過日').reset_index()           # 保留原 index
    q_keys = list(zip(q['財報民國年'], q['期別']))
    q_pass = [pd.Timestamp(d) for d in q['通過日']]
    bucket = {i: [] for i in range(len(q))}
    for _, ir in comp_inv.sort_values('法說日').iterrows():
        T = pd.Timestamp(ir['法說日'])
        ment = ir['擇要報告期']
        passed = [j for j in range(len(q)) if q_pass[j] <= T]
        future = [j for j in range(len(q)) if q_pass[j] > T]
        j_pass = passed[-1] if passed else None
        j_next = future[0] if future else None
        target, typ = None, None
        if j_pass is not None and q_keys[j_pass] in ment:
            target, typ = j_pass, '後續法說'
        elif j_next is not None and q_keys[j_next] in ment:
            target, typ = j_next, '前置法說'
        elif j_pass is not None:
            target, typ = j_pass, '後續法說'
        elif j_next is not None:
            target, typ = j_next, '前置法說'
        if target is not None:
            bucket[target].append((T, ir, typ))
    res = {}
    for pos in range(len(q)):
        idx = q.loc[pos, 'index']
        cands = sorted(bucket[pos], key=lambda x: x[0])
        if cands:
            T, ir, typ = cands[0]
            res[idx] = dict(首次法說日=T.date(), 法說時間=ir.get('召開法人說明會時間'),
                            法說地點=ir.get('召開法人說明會地點'),
                            法說擇要=ir.get('法人說明會擇要訊息'),
                            法說簡報=ir.get('法人說明會簡報內容-中文'), 法說類型=typ)
        else:
            res[idx] = dict(首次法說日=None, 法說時間=None, 法說地點=None,
                            法說擇要=None, 法說簡報=None, 法說類型=None)
    return res

inv_by_code = {c: g for c, g in inv.groupby('代號')}
all_res = {}
for code, comp_q in 季報_df.groupby(季報_df['代號'].astype(str)):
    comp_inv = inv_by_code.get(code, inv.iloc[0:0])
    all_res.update(assign_quarter(comp_q, comp_inv))

for col in ['首次法說日', '法說時間', '法說地點', '法說擇要', '法說簡報', '法說類型']:
    季報_df[col] = 季報_df.index.map(lambda i: all_res.get(i, {}).get(col))

has = 季報_df['首次法說日'].notna().sum()
print(f'各季有對應法說 {has:,} / {len(季報_df):,} 筆')
print('  前置法說', (季報_df['法說類型'] == '前置法說').sum(),
      '後續法說', (季報_df['法說類型'] == '後續法說').sum())
dup = (季報_df.dropna(subset=['首次法說日']).groupby(['代號', '首次法說日']).size())
print('  同一場法說被多季使用的筆數:', int((dup > 1).sum()))

各季有對應法說 13,431 / 33,579 筆
  前置法說 1152 後續法說 12279
  同一場法說被多季使用的筆數: 2


## 7. 交易日數計算

In [8]:
def td_signed(a, b):
    try:
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        if a > b: return -len(xtai.sessions_in_range(b, a))
        return len(xtai.sessions_in_range(a, b))
    except: return np.nan

季報_df['距截止交易日'] = 季報_df.apply(lambda r: td_signed(r['通過日'], r['截止日']), axis=1)
季報_df['通過到法說交易日'] = 季報_df.apply(
    lambda r: td_signed(r['通過日'], r['首次法說日']) if pd.notna(r['首次法說日']) else None, axis=1)
季報_df['法說到截止交易日'] = 季報_df.apply(
    lambda r: td_signed(r['首次法說日'], r['截止日']) if pd.notna(r['首次法說日']) else None, axis=1)

before = len(季報_df)
季報_df = 季報_df[季報_df['距截止交易日'].between(-100, 100)].copy()
print(f'距截止異常過濾：{before:,} → {len(季報_df):,}')

季報_df = 季報_df.rename(columns={'法說簡報': '法說簡報(中)'})
財報_df = 季報_df.copy()
print('\n欄位:', list(財報_df.columns))

距截止異常過濾：33,579 → 33,566

欄位: ['代號', '財報民國年', '期別', '簡稱', '市場別', '大型股', '截止日', '通過日', '主旨', '首次法說日', '法說時間', '法說地點', '法說擇要', '法說簡報(中)', '法說類型', '距截止交易日', '通過到法說交易日', '法說到截止交易日']


## 8. 存檔

In [9]:
財報_df.to_pickle(rf'{BASE}\財報_df.pkl')
print(f'已存 {len(財報_df):,} 筆 → 財報_df.pkl')

已存 33,566 筆 → 財報_df.pkl


## 9. Step3 涵蓋檢查：實收資本額.csv 公司是否都有資料

In [10]:
universe = set(cap_df['代號'].astype(int))
have = set(財報_df['代號'].astype(int))
missing = sorted(universe - have)
print(f'\n實收資本額母體 {len(universe)} 家，樣本涵蓋 {len(universe & have)} 家')
print(f'完全沒有資料 {len(missing)} 家：')
name_map = dict(zip(cap_df['代號'].astype(int),
                    cap_df['證券代碼'].astype(str).str.split(n=1).str[1].fillna('')))
for c in missing:
    print(f'  {c} {name_map.get(c, "")}')


實收資本額母體 1941 家，樣本涵蓋 1835 家
完全沒有資料 106 家：
  1103 嘉泥
  1110 東泥
  1210 大成
  1233 天仁
  1312 國喬
  1337 再生-KY
  1409 新纖
  1413 宏洲
  1443 立益物流
  1449 佳和
  1460 宏遠
  1466 聚隆
  1475 業旺
  1514 亞力
  1590 亞德客-KY
  1597 直得
  1604 聲寶
  1784 訊聯
  2002 中鋼
  2012 春雨
  2023 燁輝
  2035 唐榮
  2323 中環
  2348 海悅
  2351 順德
  2385 群光
  2401 凌陽
  2474 可成
  2536 宏普
  2603 長榮
  2607 榮運
  2610 華航
  2801 彰銀
  2884 玉山金
  2892 第一金
  2901 欣欣
  2910 統領
  3008 大立光
  3015 全漢
  3026 禾伸堂
  3037 欣興
  3164 景岳
  3213 茂訊
  3287 廣寰科
  3288 點晶
  3296 勝德
  3306 鼎天
  3450 聯鈞
  3481 群創
  3596 智易
  3665 貿聯-KY
  3709 鑫聯大投控
  4105 東洋
  4106 雃博
  4107 邦特
  4119 旭富
  4157 太景*-KY
  4162 智擎
  4420 光明
  4707 磐亞
  4744 皇將
  5225 東科-KY
  5227 立凱-KY
  5276 達輝-KY
  5306 桂盟
  5348 正能量智能
  5371 中光電
  5410 國眾
  5460 同協
  5465 富驊
  5514 三豐
  6144 得利影
  6170 統振
  6176 瑞儀
  6183 關貿
  6186 新潤
  6187 萬潤
  6188 廣明
  6269 台郡
  6409 旭隼
  6414 樺漢
  6441 廣錠
  6456 GIS-KY
  6525 捷敏-KY
  6569 醫揚
  6579 研揚
  6654 天正國際
  6674 鋐寶科技
  6752 叡揚
  6768 志強-KY
  6790